# 7 — RFP Parser
> Paste in a government or enterprise RFP and get back every deadline, requirement, and scoring weight — structured and ready for a bid/no-bid decision.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI


# --- Data shapes ---

class Deadline(BaseModel):
    label: str = Field(description="What this deadline is for, e.g. 'Proposal submission'")
    date: str = Field(description="Date in YYYY-MM-DD format if parseable, otherwise the raw text")
    is_hard: bool = Field(description="True if missing this deadline disqualifies the submission")


class ScoringCriterion(BaseModel):
    criterion: str = Field(description="Name of the evaluation criterion")
    weight_percent: Optional[int] = Field(
        default=None,
        ge=0,
        le=100,
        description="Percentage weight if stated in the RFP, else null",
    )
    description: str = Field(description="What the evaluators are looking for")


class Requirement(BaseModel):
    id: str = Field(description="Short identifier, e.g. 'REQ-01'")
    category: Literal["technical", "administrative", "legal", "financial"]
    text: str = Field(description="The requirement as stated")
    mandatory: bool = Field(description="True if explicitly required, False if preferred/nice-to-have")


class RFPExtraction(BaseModel):
    title: str = Field(description="Official name or title of the RFP")
    issuing_agency: str
    budget_ceiling: Optional[str] = Field(
        default=None,
        description="Maximum contract value if stated, as a string (e.g. '$500,000')",
    )
    contract_duration: Optional[str] = Field(
        default=None,
        description="Length of the contract period if stated",
    )
    deadlines: List[Deadline]
    requirements: List[Requirement]
    scoring_criteria: List[ScoringCriterion]
    summary: str = Field(description="Two-sentence plain-English summary of what is being procured")


# --- Agent ---

SYSTEM_PROMPT = SystemMessage(
    """You are a procurement analyst who extracts structured data from government and enterprise RFPs.

Given RFP text, extract:
- title: official name of the RFP
- issuing_agency: who published it
- budget_ceiling: maximum contract value if mentioned (as a string, e.g. "$500,000")
- contract_duration: how long the contract runs if mentioned
- deadlines: all dates and milestones — label, date (YYYY-MM-DD if parseable), is_hard
- requirements: all requirements — assign each an ID (REQ-01, REQ-02...), categorize as
    technical | administrative | legal | financial, and flag mandatory vs preferred
- scoring_criteria: evaluation criteria with weight_percent if stated, and what evaluators look for
- summary: two sentences plain English on what is being procured

If a field is not mentioned in the RFP, use null (for optional fields) or an empty list.
For dates, convert to YYYY-MM-DD when possible. If the year is not stated, use the context year.
Extract ALL requirements and criteria, not just a sample."""
)


def parse(rfp_text: str) -> RFPExtraction:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    parser = SYSTEM_PROMPT | llm.with_structured_output(RFPExtraction)
    return parser.invoke(rfp_text)


print("Ready.")

## The scenario

A county public health department is soliciting a vendor to build and operate a disease surveillance and outbreak reporting platform. The RFP spans several pages and buries key requirements across its scope, legal, and technical sections. The agent will surface every deadline, flag which requirements are mandatory versus preferred, and show exactly how the evaluation committee weights each section of your proposal.

In [ ]:
RFP_TEXT = """
REQUEST FOR PROPOSALS
Mecklenburg County Department of Public Health

RFP NUMBER: DPH-2026-019
TITLE: Communicable Disease Surveillance and Outbreak Reporting Platform

ISSUING AGENCY: Mecklenburg County Department of Public Health, Division of Epidemiology
CONTRACT DURATION: 2 years with one 1-year renewal option at the County's discretion

BUDGET: The County anticipates a contract ceiling not to exceed $420,000 for the
initial two-year term, inclusive of implementation, licensing, training, and support.

IMPORTANT DATES:
- RFP Release Date: 2026-05-15 (informational only)
- Vendor Questions Deadline: 2026-06-06 by 5:00 PM EST
- County Responses to Questions: 2026-06-13 (estimated)
- Proposal Submission Deadline: 2026-06-27 by 3:00 PM EST (HARD DEADLINE — late submissions disqualified)
- Oral Presentations (shortlisted vendors): 2026-07-14 through 2026-07-18
- Award Notification: 2026-07-31 (estimated)
- Contract Start: 2026-09-01

SCOPE OF WORK:
The County seeks a vendor to implement a web-based communicable disease surveillance
platform that integrates with the North Carolina Electronic Disease Surveillance System
(NC EDSS) and supports real-time outbreak alert generation, case management, and
public-facing reporting dashboards.

MANDATORY REQUIREMENTS:
REQ-01 (Technical): The platform must integrate with NC EDSS via HL7 FHIR R4 APIs.
REQ-02 (Legal): Vendor must be HIPAA-compliant and provide a signed BAA prior to contract execution.
REQ-03 (Technical): All data must be hosted within the continental United States.
REQ-04 (Administrative): Vendor must maintain a dedicated support line with a 4-hour response SLA.
REQ-05 (Financial): Vendor must provide two years of audited financial statements.
REQ-06 (Technical): The platform must support role-based access control with audit logging for all PHI access.
REQ-07 (Administrative): Vendor must provide on-site training for up to 30 County staff within 60 days of go-live.

PREFERRED QUALIFICATIONS:
REQ-08 (Technical): Prior integration with at least one other state-level disease surveillance system preferred.
REQ-09 (Administrative): Experience working with local health departments in the Southeast United States preferred.

EVALUATION CRITERIA:
1. Technical Solution and Integration Capability — 40%
   Evaluators will assess the vendor's approach to NC EDSS integration, data security architecture,
   dashboard capabilities, and the scalability of the proposed platform.

2. Relevant Experience and References — 20%
   Vendors must submit a minimum of three references from public health agencies with deployments
   of similar scope. At least one reference must be from a county or state health department.

3. Total Cost of Ownership — 25%
   Proposals should break out implementation, annual licensing, training, and optional module costs.
   The County reserves the right to negotiate final pricing.

4. Implementation Plan and Timeline — 10%
   A realistic phased rollout plan with milestones aligned to the September 2026 contract start.

5. Staff Qualifications — 5%
   Project manager must demonstrate experience leading public sector software implementations.
   Lead technical architect must hold a current certification in cloud infrastructure or security.

Proposals must be submitted electronically via the County's procurement portal at
procurement.mecknc.gov. Paper submissions will not be accepted.
"""

result = parse(RFP_TEXT)

# --- Output ---
print(f"Title:             {result.title}")
print(f"Agency:            {result.issuing_agency}")
print(f"Budget ceiling:    {result.budget_ceiling or 'Not stated'}")
print(f"Contract duration: {result.contract_duration or 'Not stated'}")
print(f"\nSummary:\n{result.summary}")

print(f"\nDEADLINES ({len(result.deadlines)})")
for d in result.deadlines:
    flag = " [HARD]" if d.is_hard else ""
    print(f"  {d.date:12}  {d.label}{flag}")

mandatory = [r for r in result.requirements if r.mandatory]
preferred = [r for r in result.requirements if not r.mandatory]
print(f"\nREQUIREMENTS — {len(mandatory)} mandatory, {len(preferred)} preferred")
for r in result.requirements:
    flag = "[mandatory]" if r.mandatory else "[preferred]"
    print(f"  {r.id}  {flag}  [{r.category}]")
    print(f"         {r.text[:90]}..." if len(r.text) > 90 else f"         {r.text}")

print(f"\nSCORING CRITERIA ({len(result.scoring_criteria)})")
for c in result.scoring_criteria:
    weight = f"{c.weight_percent}%" if c.weight_percent is not None else "N/A"
    print(f"  {weight:5}  {c.criterion}")

## Use your own data

Replace `RFP_TEXT` in the cell above with:
- Your own RFP document, pasted as a plain-text string
- Any government or enterprise solicitation — federal, state, county, or private sector

The agent will return every deadline (with hard vs. soft classification), all requirements (mandatory vs. preferred, by category), scoring weights, budget ceiling, and a two-sentence plain-English summary.